<a href="https://colab.research.google.com/github/mmilannaik/bostonhousepricing/blob/main/W16S1_SQL_Window_01Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install pandasql
!pip install pandasql --quiet

# 2. Load libraries and your CSV into a pandas DataFrame
import pandas as pd
from pandasql import sqldf

# 3. Create a helper to run SQL against any DataFrame in your notebook
pysqldf = lambda query: sqldf(query, globals())

  Preparing metadata (setup.py) ... done


**Note: Try to avoid *GROUP BY* clause to solve the problems**

For the problems use the *Health Insurance Claim* dataset. You can get the details as well as the dataset from [here](https://www.kaggle.com/datasets/thedevastator/insurance-claim-analysis-demographic-and-health).

### **Problem 1:** What are the top 5 patients who claimed the highest insurance amounts?

### **Problem 2:** What is the average insurance claimed by patients based on the number of children they have?

### **Problem 3:** What is the highest and lowest claimed amount by patients in each region?

### **Problem 4:** What is the percentage of smokers in each age group?

### **Problem 5:** What is the difference between the claimed amount of each patient and the first claimed amount of that patient?

### **Problem 6:** For each patient, calculate the difference between their claimed amount and the average claimed amount of patients with the same number of children.

### **Problem 7:** Show the patient with the highest BMI in each region and their respective rank.

### **Problem 8:** Calculate the difference between the claimed amount of each patient and the claimed amount of the patient who has the highest BMI in their region.

### **Problem 9:** For each patient, calculate the difference in claim amount between the patient and the patient with the highest claim amount among patients with the same bmi and smoker status, within the same region. Return the result in descending order difference.

### **Problem 10:** For each patient, find the maximum BMI value among their next three records (ordered by age).

### **Problem 11:** For each patient, find the rolling average of the last 2 claims.

### **Problem 12:** Find the first claimed insurance value for male and female patients, within each region order the data by patient age in ascending order, and only include patients who are non-diabetic and have a bmi value between 25 and 30.

In [2]:
insur = pd.read_csv('/content/W16S1_Task_insurance_data.csv')
insur.head(2)

,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim
0,0,1,39.0,male,23.2,91,Yes,0,No,southeast,1121.87
1,1,2,24.0,male,30.1,87,No,0,No,southeast,1131.51


In [22]:
insur['children'].value_counts()

,count
children,
0,576
1,324
2,240
3,157
4,25
5,18


# Q1 :Problem 1: What are the top 5 patients who claimed the highest insurance amounts?

In [25]:
pysqldf('''
Select *  FROM (SELECT PatientID,
RANK() OVER(ORDER BY SUM(claim) DESC) as 'rnk'
FROM insur
GROUP BY PatientID ) t
WHERE rnk<=5



''')

,PatientID,rnk
0,1340,1
1,1339,2
2,1338,3
3,1337,4
4,1336,5


# Q2 Problem 2: What is the average insurance claimed by patients based on the number of children they have?

In [29]:
pysqldf('''
SELECT children,avg_claim FROM
(
SELECT *,
AVG(claim) OVER(PARTITION BY children) AS 'avg_claim',
ROW_NUMBER() OVER(PARTITION BY children) as 'row_no'
FROM insur
) t
WHERE t.row_no =1
''')

,children,avg_claim
0,0,12327.993160
1,1,12731.171821
2,2,15073.564000
3,3,15355.318535
4,4,13850.656800
5,5,8786.035556


# Q3 : Problem 3: What is the highest and lowest claimed amount by patients in each region?

In [35]:
pysqldf('''
SELECT region,claim
FROM
(
SELECT *,
RANK() OVER(PARTITION BY region ORDER BY claim DESC) AS 'rank_region_DESC',
RANK() OVER(PARTITION BY region ORDER BY claim ASC) AS 'rank_region_ASC'

FROM insur
) t
WHERE t.rank_region_DESC = 1 OR t.rank_region_ASC =1
''')

,region,claim
0,None,1256.30
1,None,1252.41
2,northeast,58571.07
3,northeast,1694.80
4,northwest,60021.40
5,northwest,1136.40
6,southeast,63770.43
7,southeast,1121.87
8,southwest,52590.83
9,southwest,1261.44


# Q4 Problem 4: What is the percentage of smokers in each age group?